### Test Random Env with OpenAI Gymnasium

In [3]:
import gymnasium as gym
import random

In [4]:
visual_env = gym.make("CartPole-v1", render_mode='human')
training_env = gym.make("CartPole-v1")
states = visual_env.observation_space.shape[0]
actions = int(visual_env.action_space.n)

In [5]:
actions

2

In [6]:
episodes = 10
for episode in range(1, episodes + 1):
    state = visual_env.reset()
    done = False
    score = 0

    while not done:
        visual_env.render()
        action = random.choice([0,1])
        n_state, reward, terminated, truncated, info = visual_env.step(action)
        score += reward
        done = terminated or truncated
    print(f"Episode {episode}: Score: {score}")

Episode 1: Score: 37.0
Episode 2: Score: 11.0
Episode 3: Score: 18.0
Episode 4: Score: 33.0
Episode 5: Score: 17.0
Episode 6: Score: 21.0
Episode 7: Score: 25.0
Episode 8: Score: 18.0
Episode 9: Score: 27.0
Episode 10: Score: 20.0


### Create Deep Learning Model with Keras

In [7]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Input
from tensorflow.keras.optimizers import Adam

In [8]:
def build_model(states, actions):
    model = Sequential()
    model.add(Input(shape=(1, states)))
    model.add(Flatten())
    model.add(Dense(24, activation='relu'))
    model.add(Dense(24, activation='relu'))
    model.add(Dense(actions, activation='linear'))
    return model

In [9]:
model = build_model(states, actions)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 4)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 24)             │           120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │           600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            50 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 770 (3.01 KB)

 Trainable params: 770 (3.01 KB)

 Non-trainable params: 0 (0.00 B)

### Build Agent with Keras-RL

#### keras-rl2 compatibility note

`keras-rl2==1.0.5` predates the current Python/TensorFlow/Keras/Gymnasium stack used by this notebook. I patched this package in my virtual environment locally, but for you these patches will not be present.

If you install `keras-rl2` into a new virtual environment and run into the errors below, either patch the installed `rl` package in that environment or use an older dependency set where `keras-rl2` is known to work.

Patched library issues (you need to patch these yourself):

- Import failure: Keras 3 removed `model_from_config`, so `rl.util.clone_model` now uses Keras 3's supported model cloning API.
- Import side effect: `rl.agents.ddpg` disabled eager execution globally during import, which broke Keras 3 weight cloning; that import-time side effect was removed.
- Callback mismatch: `rl.callbacks` used private `tensorflow.python.keras` callback classes; it now uses public Keras callback and progress bar imports.
- DQN model access: old DQN code assumed `model.input`, `model.output`, and unconditional `reset_states()` behavior; it was patched for Keras 3 Sequential/Functional models and non-stateful models.
- Gymnasium API: `reset()` and `step()` return newer tuple shapes; `rl.core` now adapts Gymnasium results to the old Gym shape expected by `keras-rl2`.
- DQN state batches: state batches are normalized to numeric arrays so TensorFlow can convert them during prediction/training.
- Soft target updates: Keras 3 no longer supports the old optimizer `get_updates` path used by `keras-rl2`; DQN soft target updates are applied after train batches instead.
- Training logs: Keras 3 returns `loss`, `mae`, and `mean_q` values during updates; DQN metric names were patched so interval logging does not crash with an inhomogeneous metrics array.
- Long runs: `Agent.fit()` used `np.int16` counters, which overflow before 50,000 steps; training counters now use normal Python integers.
- Visualization: the visualizer called `env.render(mode='human')`, but Gymnasium selects render mode when the environment is created; the visualizer now falls back to `env.render()`.

After applying or reapplying any library patch, restart the notebook kernel before running the cells below.

In [10]:
from rl.agents import DQNAgent
from rl.policy import BoltzmannQPolicy
from rl.memory import SequentialMemory

In [11]:
def build_agent(model, actions):
    policy = BoltzmannQPolicy()
    memory = SequentialMemory(limit=50000, window_length=1)
    dqn = DQNAgent(
        model=model,
        memory=memory,
        policy=policy,
        nb_actions=actions,
        nb_steps_warmup=1000,
        target_model_update=1e-2
    )
    return dqn

In [12]:
dqn = build_agent(model, actions)
dqn.compile(Adam(learning_rate=1e-3), metrics=['mae'])
dqn.fit(training_env, nb_steps=50000, visualize=False, verbose=1)

Training for 50000 steps ...
Interval 1 (0 steps performed)
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 12s 1ms/step - reward: 1.0000
137 episodes - episode_reward: 72.073 [9.000, 360.000] - loss: 1.340 - mae: 9.297 - mean_q: 18.675

Interval 2 (10000 steps performed)
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 13s 1ms/step - reward: 1.0000
44 episodes - episode_reward: 224.318 [160.000, 353.000] - loss: 3.209 - mae: 25.155 - mean_q: 50.974

Interval 3 (20000 steps performed)
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 13s 1ms/step - reward: 1.0000
47 episodes - episode_reward: 216.574 [167.000, 345.000] - loss: 3.976 - mae: 34.708 - mean_q: 70.224

Interval 4 (30000 steps performed)
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 14s 1ms/step - reward: 1.0000
45 episodes - episode_reward: 219.822 [149.000, 316.000] - loss: 3.935 - mae: 38.894 - mean_q: 78.594

Interval 5 (40000 steps performed)
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 14s 1ms/step - reward: 1.0000
done, took 66.008 seconds


In [13]:
scores = dqn.test(training_env, nb_episodes=100, visualize=False)
print(np.mean(scores.history['episode_reward']))

Testing for 100 episodes ...
Episode 1: reward: 500.000, steps: 500
Episode 2: reward: 500.000, steps: 500
Episode 3: reward: 500.000, steps: 500
Episode 4: reward: 500.000, steps: 500
Episode 5: reward: 500.000, steps: 500
Episode 6: reward: 500.000, steps: 500
Episode 7: reward: 500.000, steps: 500
Episode 8: reward: 313.000, steps: 313
Episode 9: reward: 500.000, steps: 500
Episode 10: reward: 265.000, steps: 265
Episode 11: reward: 500.000, steps: 500
Episode 12: reward: 500.000, steps: 500
Episode 13: reward: 500.000, steps: 500
Episode 14: reward: 500.000, steps: 500
Episode 15: reward: 500.000, steps: 500
Episode 16: reward: 500.000, steps: 500
Episode 17: reward: 500.000, steps: 500
Episode 18: reward: 500.000, steps: 500
Episode 19: reward: 500.000, steps: 500
Episode 20: reward: 500.000, steps: 500
Episode 21: reward: 500.000, steps: 500
Episode 22: reward: 500.000, steps: 500
Episode 23: reward: 500.000, steps: 500
Episode 24: reward: 500.000, steps: 500
Episode 25: reward: 

In [14]:
_ = dqn.test(visual_env, nb_episodes=10, visualize=True)

Testing for 10 episodes ...
Episode 1: reward: 500.000, steps: 500
Episode 2: reward: 500.000, steps: 500
Episode 3: reward: 500.000, steps: 500
Episode 4: reward: 500.000, steps: 500
Episode 5: reward: 500.000, steps: 500
Episode 6: reward: 500.000, steps: 500
Episode 7: reward: 500.000, steps: 500
Episode 8: reward: 500.000, steps: 500
Episode 9: reward: 500.000, steps: 500
Episode 10: reward: 500.000, steps: 500


### Reloading Agent from Memory

In [15]:
dqn.save_weights('dqn_cart_pole_v1.weights.h5', overwrite=True)

In [16]:
del model
del dqn
del visual_env
del training_env

In [19]:
dqn.test(visual_env, nb_episodes=5, visualize=True)

NameError: name 'dqn' is not defined

In [20]:
visual_env = gym.make('CartPole-v1', render_mode='human')
actions = int(visual_env.action_space.n)
states = visual_env.observation_space.shape[0]
model = build_model(states, actions)
dqn = build_agent(model, actions)
dqn.compile(Adam(learning_rate=1e-3), metrics=['mae'])

In [21]:
dqn.load_weights("dqn_cart_pole_v1.weights.h5")

In [22]:
_ = dqn.test(visual_env, nb_episodes=5, visualize=True)

Testing for 5 episodes ...
Episode 1: reward: 500.000, steps: 500
Episode 2: reward: 500.000, steps: 500
Episode 3: reward: 500.000, steps: 500
Episode 4: reward: 500.000, steps: 500
Episode 5: reward: 500.000, steps: 500
